# IPL Match Predictor — Algorithm Walkthrough (Notebook)

This notebook is the **runnable companion** to [`docs/ALGORITHM_WALKTHROUGH.md`](../docs/ALGORITHM_WALKTHROUGH.md). It walks through the prediction algorithm end-to-end on two real IPL 2026 matches:

1. **M71 RCB vs GT** (Qualifier 1) — ensemble picked GT 64%, RCB won by 92 runs → **WRONG**
2. **M10 SRH vs LSG** — ensemble picked SRH 61%, LSG chased successfully → **WRONG**

**Important:** This notebook bypasses `src.models.predict_match()` which has a known bug — it builds a 17-feature dict but the trained models expect 46 features, silently falls back to pure Elo, and produces inflated numbers. We call `ensemble_predict_proba()` directly with the properly-shaped feature row. See the markdown doc's honesty note for details.

**To run:** Restart kernel → Run All. Total runtime ~30 seconds.

**The bigger picture:** Both case studies show the model getting it wrong. End-to-end accuracy is **48.6%** on IPL 2026 — barely better than random. The hand-curated archive at 61.4% substantially outperforms the trained model. This is intentional — the gap is the headroom for a learner to add better features.

**Hack-along ideas:**
- Change `predict_by_lookup()` arguments to predict any match
- Modify features in `src/features.py` and re-run cells 4 onward
- Compare per-model scores (RF/XGB/GB/LR) to spot where the ensemble disagrees

## Setup — imports + path

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

# Find repo root (one level up from notebooks/)
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd

from src.data_loader import load_matches, clean_matches, clean_team_names
from src.features import compute_elo_ratings, engineer_features
from src.models import build_ensemble, predict_match, prepare_features, split_data

print('Repo root:', REPO)
print('All imports OK')

## Stage 1-2 — Load + clean

Load historical IPL matches (2008-2025) plus the IPL 2026 export. Combined, sorted chronologically, team names standardized.

In [ ]:
historical = load_matches()
season_2026 = pd.read_csv(REPO / 'data' / 'raw' / 'matches_2026.csv')
common = [c for c in historical.columns if c in season_2026.columns]
combined = pd.concat([historical[common], season_2026[common]], ignore_index=True)
combined['date'] = pd.to_datetime(combined['date'], errors='coerce')
combined = combined.sort_values('date').reset_index(drop=True)
combined, _ = clean_team_names(combined)
combined = clean_matches(combined)
# Fill missing toss data so prepare_features doesn't drop rows (2026 export has toss
# data for only 12 of 71 completed matches — without this fill, 58 of 70 rows vanish)
combined['toss_winner'] = combined['toss_winner'].fillna(combined['team1'])
combined['toss_decision'] = combined['toss_decision'].fillna('field')

print(f'Total matches: {len(combined)}')
print(f'Date range: {combined["date"].min().date()} → {combined["date"].max().date()}')
print(f'IPL 2026 matches: {(combined["season"].astype(str) == "2026").sum()}')
combined.head(3)

## Stage 3 — Elo ratings (chronological)

Walks through every match in date order and updates each team's Elo with K=32. After this cell, every row in `combined` has `elo_team1` and `elo_team2` populated with the **pre-match** ratings.

**Elo intuition:** Each match nudges both teams. Beating a stronger team gains you more points than beating a weaker one. Over 1,000+ matches the ratings stabilize to reflect each team's strength.

In [ ]:
combined = compute_elo_ratings(combined)

# Show RCB's Elo evolution across 2026 leading into M71
rcb_2026 = combined[
    (combined['date'] >= '2026-03-01')
    & (combined['date'] <= '2026-05-26')
    & ((combined['team1'] == 'Royal Challengers Bangalore') | (combined['team2'] == 'Royal Challengers Bangalore'))
].copy()
rcb_2026['rcb_elo_pre_match'] = np.where(
    rcb_2026['team1'] == 'Royal Challengers Bangalore',
    rcb_2026['elo_team1'], rcb_2026['elo_team2']
)
print('RCB Elo evolution through IPL 2026 (pre-match ratings):')
rcb_2026[['date', 'team1', 'team2', 'rcb_elo_pre_match', 'winner']].tail(15)

## Stage 4 — Feature engineering

Adds 10 more features: momentum (last-5 win rate), H2H winrate + sample size, home flags, toss flags, venue chase bias, plus 3 interaction features.

In [ ]:
combined = engineer_features(combined)
print(f'Total features available: {len(combined.columns)}')
print('\nKey engineered features:')
for f in ['elo_team1', 'elo_team2', 'elo_diff', 'elo_expected',
          'momentum_team1', 'momentum_team2', 'momentum_diff',
          'h2h_team1_winrate', 'h2h_matches',
          'home_team1', 'home_team2',
          'toss_winner_is_team1', 'toss_chose_field',
          'venue_chase_bias']:
    print(f'  {f}')

## Stage 5 — Train the 4-model ensemble (once, on pre-2026 data)

Train on all matches with date < 2026-01-01. The 4 models (RF + XGB + GB + LR) are wrapped in `CalibratedClassifierCV` for well-calibrated probabilities. Each match contributes a `recency_weight` so recent seasons influence the model more than 2008.

In [ ]:
import sklearn

# Prepare features ON THE COMBINED DATASET (one-hot + scaling) so 2026 rows have the
# same 46-column schema the models will be trained on. Slight scaler leakage from 2026
# into the training transform — marginal effect (72 of 1241 rows). The strictly leak-free
# version would walk-forward, refitting for each match (~30 min runtime).
model_df, feature_names = prepare_features(combined)
model_df = model_df.astype(np.float64)
print(f'Full feature matrix: {model_df.shape[0]} rows × {model_df.shape[1]} features')

# Training subset: pre-2026 with known winner
cutoff = pd.Timestamp('2026-01-01')
train_mask = (combined['date'] < cutoff) & combined['winner'].notna()
train_idx = combined[train_mask].index.intersection(model_df.index)
X_train = model_df.loc[train_idx]
y_train = (combined.loc[train_idx, 'winner'] == combined.loc[train_idx, 'team1']).astype(np.int64)
sw = combined.loc[train_idx, 'recency_weight'].values.astype(np.float64)

print(f'Training set: {len(X_train)} matches ({combined.loc[train_idx, "date"].min().date()} → {combined.loc[train_idx, "date"].max().date()})')

# sklearn 1.4 has a dtype bug inside CalibratedClassifierCV → _sigmoid_calibration.
# Fall back to uncalibrated on older sklearn (probabilities slightly less calibrated
# but algorithm walkthrough still illustrates the full pipeline).
sk_version = tuple(int(x) for x in sklearn.__version__.split('.')[:2])
use_calibration = sk_version >= (1, 5)
if not use_calibration:
    print(f'[WARN] sklearn {sklearn.__version__} detected — disabling calibration (needs >= 1.5)')

import time; t0 = time.time()
models = build_ensemble(X_train, y_train, sample_weight=sw, calibrate=use_calibration)
print(f'\nTrained 4-model ensemble in {time.time()-t0:.1f}s (calibrated={use_calibration})')
print(f'Models trained: {list(models.keys())}')

## Helper: predict one match by lookup

Builds the feature vector for any match and runs the ensemble. Returns the pick, confidence, per-model scores, and feature snapshot.

In [ ]:
# Predict a match by looking up its row from `combined`, extracting its prepared feature
# vector from `model_df`, and calling each of the 4 ensemble models directly.
# This BYPASSES src.models.predict_match() which has a known bug: it builds a 17-feature
# dict but the trained models expect 46 features. The exception is caught and the function
# silently falls back to pure Elo expectation — see the markdown walkthrough's honesty note.
def predict_by_lookup(date_str, t1, t2):
    rows = combined[(combined['date'] == pd.Timestamp(date_str))
                    & (combined['team1'] == t1) & (combined['team2'] == t2)]
    if len(rows) == 0:
        rows = combined[(combined['date'] == pd.Timestamp(date_str))
                        & (combined['team1'] == t2) & (combined['team2'] == t1)]
    row = rows.iloc[0]
    orig_idx = row.name
    if orig_idx not in model_df.index:
        raise ValueError(f'Row {orig_idx} dropped during prepare_features (probably bad data)')
    X_row = model_df.loc[[orig_idx]]

    # Per-model predictions
    weights = models['weights']
    model_scores = {}
    ensemble_prob = 0.0
    for name, w in weights.items():
        p = float(models[name].predict_proba(X_row)[:, 1][0])
        model_scores[name] = round(p * 100, 1)
        ensemble_prob += w * p

    pick = row['team1'] if ensemble_prob >= 0.5 else row['team2']
    confidence = round(ensemble_prob*100) if ensemble_prob >= 0.5 else round((1-ensemble_prob)*100)
    return row, {
        'winner': pick,
        'confidence': confidence,
        'team1_prob': round(ensemble_prob*100, 1),
        'team2_prob': round((1-ensemble_prob)*100, 1),
        'model_scores': model_scores,
    }

## Case 1 — M71 RCB vs GT (Qualifier 1, 26 May 2026)

Top-of-table RCB (20 pts) vs runners-up GT (18 pts) at neutral playoff venue HPCA Dharamshala.

In [ ]:
row71, pred71 = predict_by_lookup('2026-05-26', 'Royal Challengers Bangalore', 'Gujarat Titans')

print(f"Match:  {row71['team1']} vs {row71['team2']}")
print(f"Venue:  {row71['venue']}")
print(f"Toss:   {row71.get('toss_winner','?')} chose to {row71.get('toss_decision','?')}")
print(f"Actual: {row71.get('winner','?')}\n")

print('--- Feature snapshot (pre-match) ---')
for f in ['elo_team1','elo_team2','elo_diff','elo_expected',
         'momentum_team1','momentum_team2','momentum_diff',
         'h2h_team1_winrate','h2h_matches',
         'home_team1','home_team2','toss_winner_is_team1','toss_chose_field','venue_chase_bias']:
    v = row71.get(f)
    print(f'  {f:>25}: {v}')

print('\n--- Per-model probabilities P(team1 wins) ---')
for m, s in pred71['model_scores'].items():
    w = models['weights'][m]
    print(f'  {m.upper():>4} (weight {w:.2f}): {s}%')
print(f"\n  Weighted ensemble: {pred71['team1_prob']}% team1 / {pred71['team2_prob']}% team2")
print(f"  Pick:       {pred71['winner']}")
print(f"  Confidence: {pred71['confidence']}%")

correct = pred71['winner'] == row71.get('winner')
print(f"\n  Outcome:    {'[OK] CORRECT' if correct else '[FAIL] WRONG'}")

### Reading Case 1

The four trained models do NOT converge on the same answer (as they did under the buggy `predict_match()` fallback). RF says 43%, XGB strongly leans GT at 29%, GB is near a coin flip at 46%, LR strongly leans GT at 23%. Weighted ensemble lands at **GT 64%** — but RCB won by 92 runs.

**Why the model was wrong:** Elo was nearly tied. GT's +20% momentum edge nudged the tree models toward GT. The model has no concept of "playoff intensity" or "Kohli's Orange Cap form" — both of which the hand-curated archive flagged as RCB-favoring signals.

## Case 2 — M10 SRH vs LSG (5 Apr 2026)

SRH at home, LSG won toss + chose to field.

In [ ]:
row10, pred10 = predict_by_lookup('2026-04-05', 'Sunrisers Hyderabad', 'Lucknow Super Giants')

print(f"Match:  {row10['team1']} vs {row10['team2']}")
print(f"Venue:  {row10['venue']}")
print(f"Toss:   {row10.get('toss_winner','?')} chose to {row10.get('toss_decision','?')}")
print(f"Actual: {row10.get('winner','?')}\n")

print('--- Feature snapshot (pre-match) ---')
for f in ['elo_team1','elo_team2','elo_diff','elo_expected',
         'momentum_team1','momentum_team2','momentum_diff',
         'h2h_team1_winrate','h2h_matches',
         'home_team1','home_team2','toss_winner_is_team1','toss_chose_field','venue_chase_bias']:
    v = row10.get(f)
    print(f'  {f:>25}: {v}')

print('\n--- Per-model probabilities P(team1 wins) ---')
for m, s in pred10['model_scores'].items():
    w = models['weights'][m]
    print(f'  {m.upper():>4} (weight {w:.2f}): {s}%')
print(f"\n  Weighted ensemble: {pred10['team1_prob']}% team1 / {pred10['team2_prob']}% team2")
print(f"  Pick:       {pred10['winner']}")
print(f"  Confidence: {pred10['confidence']}%")

correct = pred10['winner'] == row10.get('winner')
print(f"\n  Outcome:    {'[OK] CORRECT' if correct else '[FAIL] WRONG'}")

### Reading Case 2 — post-mortem on the wrong call

The model leaned hard on:
- **Elo +34** (SRH stronger overall)
- **Momentum +60%** (SRH on a 4-of-5 streak, LSG on 1-of-5)
- **Home advantage** (SRH at Rajiv Gandhi)

And produced **SRH 68%** — a high-confidence call.

But LSG actually won by 5 wickets. Where did the model go wrong?

**H2H said LSG had won 4 of the prior 6 meetings (33% SRH winrate)**, but the model heavily discounts H2H when the sample is small (N<10). Combined with the toss-and-bowl decision (chase advantage at a neutral-chase-bias venue), LSG had structural matchup edges that the model's feature mix didn't capture.

**Learner exercise:** Open [`src/features.py`](../src/features.py) and modify `engineer_features()` to up-weight `h2h_team1_winrate` when `h2h_matches >= 5`. Rerun the notebook. Does the M10 prediction flip? What happens to overall 2026 accuracy?

## Try your own match

Change the parameters in `predict_by_lookup()` and predict any IPL 2026 match. Full team names required (e.g., `Royal Challengers Bangalore`, not `RCB`).

In [ ]:
# Edit these values and re-run
MATCH_DATE = '2026-05-22'  # M67
TEAM1 = 'Sunrisers Hyderabad'
TEAM2 = 'Royal Challengers Bangalore'

row, pred = predict_by_lookup(MATCH_DATE, TEAM1, TEAM2)
print(f"{row['team1']} vs {row['team2']} @ {row['venue']}")
print(f"  Pick: {pred['winner']} ({pred['confidence']}%)")
print(f"  Actual: {row.get('winner','?')}")
print(f"  Outcome: {'[OK]' if pred['winner'] == row.get('winner') else '[FAIL]'}")

## Run the full 2026 season

For the complete season run with summary table:

```bash
python scripts/predict_all_2026.py
```

Expected: **34 / 70 correct = 48.6% accuracy** (barely better than a coin flip). 2 matches unresolved (M12 washout, M72 upcoming).

## The hard truth about this pipeline's accuracy

| Approach | Accuracy on IPL 2026 |
|----------|----------------------|
| Random coin flip | 50.0% |
| Trained 4-model ensemble (this notebook) | **48.6%** |
| Hand-curated archive (expert overlay) | 61.4% |

The trained ensemble underperforms even a coin flip. The hand-curated overlay (with contextual reasoning on top of similar features) beats the model by ~13 percentage points. **There is real headroom here for any learner who wants to add better features.**

## Next steps for learners

1. **Run the pipeline**, confirm the 48.6% number
2. **Find 3 high-confidence wrong picks** (where conf ≥ 70% but outcome was wrong, e.g., M11 RCB-vs-CSK 78%, M22 CSK-vs-KKR 72%, M23 RCB-vs-LSG 76%) — read each match's features, hypothesize why
3. **Add a new feature** to `src/features.py` (e.g., dew indicator, opening pace matchup, day/night flag, captain-change indicator)
4. **Re-run** — did accuracy go up?
5. **Fix `predict_match()`** — the 17-vs-46 feature mismatch + silent fallback is a real bug worth fixing
6. **Try to beat 61.4%** — that's the hand-curated archive in [`predictions-2026/`](../predictions-2026/), and a fair stretch goal